Helper JSON Functions

In [6]:
import json 
import json
from pathlib import Path

def save_json(data: dict, filepath: str, indent: int = 4) -> None:
    """
    Save a Python dictionary (or any JSON-serializable object) to a JSON file.

    Args:
        data (dict): The dictionary or object to serialize.
        filepath (str): Path to the output JSON file.
        indent (int, optional): Number of spaces for indentation. Defaults to 4.

    Notes:
        - Automatically creates directories if they don't exist.
        - Uses UTF-8 encoding and keeps non-ASCII characters readable.
        - Falls back to str() for non-serializable objects.
    """
    path = Path(filepath)
    path.parent.mkdir(parents=True, exist_ok=True)

    # pretty-print JSON with compact inline lists
    json_str = json.dumps(
        data,
        ensure_ascii=False,
        indent=4
    )

    with open(path, "w", encoding="utf-8") as f:
        f.write(json_str)

    print(f"✅ JSON saved to: {path.resolve()}")

def load_json(file_name, parse_int=int, parse_float=float):
    with open(file_name, 'r') as file:
        data = json.load(file, parse_int=parse_int, parse_float=parse_float)
    return data

def extract_json_fast(text: str):
    """
    Efficiently extract the outermost JSON object from an LLM response.
    Works in O(n) time with minimal regex overhead.
    Returns the parsed JSON object or None if parsing fails.
    """
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or start >= end:
        return None

    json_str = text[start:end+1]

    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        # Lightweight fallback: fix common trailing commas or quotes if needed
        try:
            cleaned = json_str.replace("\n", "").replace("\r", "")
            return json.loads(cleaned)
        except json.JSONDecodeError:
            return None

In [7]:
node_type_to_vector_schema = load_json('node_type_to_vector_schema.json')
node_id_to_document_json = load_json('node_id_to_document.json')
node_type_to_meta_schema = load_json('node_type_to_meta_schema.json')
configs = load_json('configs.json')

In [8]:
data_split = 'test'
dataset_name = 'prime'
llm_model = 'meta/llama-3.3-70b-instruct'
emb_model = 'text-embedding-ada-002'
configs_path = 'configs.json'

In [9]:
from stark_qa import load_skb, load_qa
from llm_bridge_updated import LlmBridge
from settings import Settings

dataset_name = 'prime'
qa_dataset = load_qa(dataset_name)
kb = load_skb(dataset_name, download_processed=True)
llm_bridge = LlmBridge(llm_model, configs_path, None)
settings = Settings(dataset_name, llm_model=llm_model, emb_model=emb_model, configs_path=configs_path)

Use file from /home/aditya/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/qa/prime/stark_qa/stark_qa_human_generated_eval.csv.
Loading from /home/aditya/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/skb/prime/processed!
Using Nvidia SIM model: meta/llama-3.3-70b-instruct


In [17]:
alpha = settings.get("alpha")
nodes_to_consider = str(kb.node_type_lst()).replace("'","")
edges_to_consider = str(list(settings.configs.get("edge_type_long2short").keys())).replace("'","")
properties_to_consider = str(list(settings.configs.get("avail_node_properties").keys())).replace("'","")
# qa_dataset = eval_data
query_tests = list(map(int, list(qa_dataset.get_idx_split()['test-0.1'])))

In [22]:
query_tests_10 = query_tests[:10]
query_tests_10

[9, 26, 88, 195, 231, 245, 252, 253, 256, 320]

In [25]:
responses = {}
for query_id in query_tests_10:
    query = qa_dataset[query_id][0]
    prompt = (
        f"From the following query Q, extract all entities and classify each by type. "
        f"For every entity, also extract a comprehensive context that fully captures all relevant information "
        f"from Q used to infer and understand that entity — including its role, descriptive details, related conditions, "
        f"and how it fits semantically in the query. The context must be self-contained so that it can be understood "
        f"without needing Q again (no information loss allowed).\n\n"
        f"Follow these rules precisely:\n"
        f"1. Exactly one entity must be labeled as 'ANSWER'.\n"
        f"2. Other entities should be labeled sequentially as A, B, C, etc.\n"
        f"3. If an entity can belong to multiple types, list all possible types as a JSON array "
        f"(e.g., 'hypertension': ['disease', 'effect/phenotype']).\n"
        f"4. Each 'context' field must contain all the descriptive or relational details from Q "
        f"that define or justify the entity’s inclusion.\n"
        f"5. Do not omit any detail that could later affect relation or constraint extraction.\n"
        f"6. Output only valid JSON — no text or explanation outside it.\n\n"
        f"Q: {query}\n\n"
        f"Available entity types: {nodes_to_consider}\n\n"
        f"Output strictly in this JSON format:\n"
        f'{{\n'
        f'  "A": {{ "types": ["type_1", "type_2", ...], "context": "<self-contained, descriptive context>" }},\n'
        f'  "B": {{ "types": ["type_1", ...], "context": "<self-contained, descriptive context>" }},\n'
        f'  "ANSWER": {{ "types": ["type_1", ...], "context": "<self-contained, descriptive context>" }}\n'
        f'}}'
    )



    response = llm_bridge.ask_llm(prompt)[0]
    responses[query_id] = json.loads(response)
responses

{9: {'A': {'types': ['pathway', 'biological_process'],
   'context': 'DCC-mediated attractive signaling, a process involving the interaction of specific molecules to guide cell movement or other cellular activities'},
  'B': {'types': ['cellular_component'],
   'context': 'actin filaments, which are major components of the cytoskeleton and play a crucial role in cell shape, division, and movement'},
  'C': {'types': ['cellular_component', 'gene/protein'],
   'context': 'actin-binding LIM protein family, a group of proteins characterized by their ability to bind to actin and containing LIM domains, which are involved in protein-protein interactions'},
  'ANSWER': {'types': ['gene/protein'],
   'context': 'a gene or protein that is engaged in DCC-mediated attractive signaling, capable of binding to actin filaments, and belongs to the actin-binding LIM protein family, indicating its role in cellular signaling and cytoskeletal organization'}},
 26: {'A': {'types': ['anatomy'],
   'context'

In [45]:
if 'responses' in locals() or 'responses' in globals():
    save_json(responses, f'entity_extraction_responses_{data_split}_10.json')
else:
    responses = load_json(f'entity_extraction_responses_{data_split}_10.json')

✅ JSON saved to: /home/aditya/BtechProject/BtechProject/FocusedRetriever/entity_extraction_responses_test_10.json


In [36]:
from difflib import get_close_matches

def create_prompt_with_schema_context(entities, query):
    entity_types = set()
    for entity, entity_details in entities.items():
        entity_types |= set(entity_details['types'])
    
    known_types = kb.node_type_lst()
    schema_filtered = {}
    missing_types = {}

    for etype in entity_types:
        if etype in node_type_to_meta_schema:
            schema_filtered[etype] = node_type_to_meta_schema[etype]
        else:
            # find closest match (case-insensitive)
            best_match = get_close_matches(etype.lower(), [t.lower() for t in known_types], n=1, cutoff=0.4)
            if best_match:
                # find actual case-sensitive match in known_types
                matched_type = next(t for t in known_types if t.lower() == best_match[0])
                schema_filtered[etype] = node_type_to_meta_schema.get(matched_type, {})
                missing_types[etype] = matched_type
            else:
                missing_types[etype] = None  # no close match found

    if missing_types:
        print(f"[Info] Missing or remapped schema types:")
        for k, v in missing_types.items():
            if v:
                print(f"  - '{k}' → matched to '{v}'")
            else:
                print(f"  - '{k}' → no suitable match found")

    prompt = f"""
  You are a biomedical information extraction assistant.

  INPUTS:
  1. Entities (already identified, with possible types, constant flag, and their rich contextual descriptions from the previous step):
  {entities}

  2. Relevant type schemas (each schema lists valid fields for that type):
  {schema_filtered}

  3. Relation type compatibility:
  - ppi: gene/protein ↔ gene/protein
  - carrier: gene/protein ↔ drug
  - enzyme: drug → gene/protein (enzyme)
  - target: drug → gene/protein
  - transporter: gene/protein ↔ drug
  - indication: drug → disease
  - contraindication: drug → disease
  - off-label use: drug → disease
  - synergistic interaction: drug ↔ drug
  - associated with: gene/protein → effect/phenotype OR gene/protein → disease
  - parent-child: disease ↔ disease
  - phenotype present: effect/phenotype → disease
  - phenotype absent: effect/phenotype → disease
  - side effect: effect/phenotype ← drug
  - interacts with: generic interaction
  - linked to: exposure → disease
  - expression present: gene/protein → anatomy
  - expression absent: gene/protein → anatomy

  OBJECTIVE:
  From the entities and their contextual information, extract:
  1. **Field-level structured details** for each entity according to its schema.
  2. **Relations** between entities that are explicitly or clearly implied by the context.
  3. **Constant flags** indicating if the entity represents a fixed, known biological concept (true) or a general/abstract placeholder (false).

  DETAILED INSTRUCTIONS:
  - For each entity:
    - Keep its `"constant"` value as provided in the input.
    - For each possible type:
      - Extract only schema fields that are **explicitly stated or strongly implied** in the context.
      - Include narrative, descriptive, or quantitative values that support the field.
      - Omit empty or irrelevant fields.
  - For relationships:
    - Infer relations only when there is clear textual or contextual evidence.
    - Relation direction and compatibility must match the biomedical mapping above.
    - If no valid relation is found, omit it.
    - Do not infer unsupported or ambiguous relations.

  OUTPUT FORMAT (strict JSON, no extra text):
  {{
    "entities": {{
      "EntityName": {{
        "constant": true/false,
        "types": {{
          "TypeName": {{
            "field_1": "value_1",
            "field_2": "value_2",
            ...
          }}
        }}
      }},
      ...
    }},
    "relations": [
      {{
        "source": "EntityKey",
        "target": "EntityKey",
        "type": "relation_type"
      }},
      ...
    ]
  }}

  IMPORTANT:
  - JSON must be valid and well-formed.
  - Include `"relations": []` if none exist.
  - Use only information found or implied in the provided context.
  - Avoid information loss — every relevant contextual detail should be represented through fields, relations, or semantics.
  - Do NOT output any explanatory text or commentary outside JSON.
  """

    return prompt


In [44]:
responses_constraints = {}
for query_id in query_tests_10:
    query_id = int(query_id)
    prompt_get_constraints = create_prompt_with_schema_context(responses[query_id], qa_dataset[query_id][0])
    response = extract_json_fast(llm_bridge.ask_llm(prompt_get_constraints)[0])
    responses_constraints[query_id] = response
    

In [46]:
if 'responses_constraints' in locals() or 'responses_constraints' in globals():
    save_json(responses_constraints, f'entity_constraints_extraction_responses_{data_split}_10.json')
else:
    responses_constraints = load_json(f'entity_constraints_extraction_responses_{data_split}_10.json')

✅ JSON saved to: /home/aditya/BtechProject/BtechProject/FocusedRetriever/entity_constraints_extraction_responses_test_10.json


In [47]:
responses_constraints

{9: {'entities': {'A': {'constant': False,
    'types': {'pathway': {'name': 'DCC-mediated attractive signaling',
      'biological_mechanism': 'cell movement or other cellular activities',
      'molecular_context': 'interaction of specific molecules'},
     'biological_process': {'name': 'DCC-mediated attractive signaling',
      'functional_summary': 'a process involving the interaction of specific molecules to guide cell movement or other cellular activities'}}},
   'B': {'constant': False,
    'types': {'cellular_component': {'name': 'actin filaments',
      'functional_summary': 'major components of the cytoskeleton and play a crucial role in cell shape, division, and movement'}}},
   'C': {'constant': False,
    'types': {'cellular_component': {'name': 'actin-binding LIM protein family'},
     'gene/protein': {'name': 'actin-binding LIM protein family',
      'functional_summary': 'a group of proteins characterized by their ability to bind to actin and containing LIM domains, wh